# TRF5 — Tribunal Regional Federal da 5ª Região

Public process consultation (`cpopg`) for the federal courts under the
fifth region (AL, CE, PB, PE, RN, SE), via the PJe `ConsultaPublica`
system at
[pje1g.trf5.jus.br/pjeconsulta/](https://pje1g.trf5.jus.br/pjeconsulta/ConsultaPublica/listView.seam).

| Feature | Available |
|---------|-----------|
| cpopg   | Yes       |
| cposg   | No        |
| cjsg    | No        |
| cjpg    | No        |

## Looking up a single process

In [1]:
import juscraper as jus

trf5 = jus.scraper("trf5")

In [2]:
df = trf5.cpopg("0058457-31.2025.4.05.8000")
print(df.shape)
df[["id_cnj", "processo", "classe", "orgao_julgador"]]

TRF5 cpopg: 100%|██████████| 1/1 [00:01<00:00,  1.21s/it]

(1, 12)


,id_cnj,processo,classe,orgao_julgador
0,00584573120254058000,0058457-31.2025.4.05.8000,RECLAMAÇÃO PRÉ-PROCESSUAL (11875),Ambiente do Centro de Conciliação de Maceió


## Available columns

In [3]:
df.columns.tolist()

['processo',
 'data_distribuicao',
 'classe',
 'assunto',
 'jurisdicao',
 'orgao_julgador',
 'endereco_orgao',
 'polo_ativo',
 'polo_passivo',
 'movimentacoes',
 'documentos',
 'id_cnj']

The first three columns are the canonical scalars; the trailing four are
list-typed and carry the nested arrays (parties, movements, attached
documents).

## Inspecting movements

In [4]:
movs = df.iloc[0]["movimentacoes"]
print(f"{len(movs)} events recorded")
for m in movs[:5]:
    print(f"  {m['data']} - {m['descricao']}")

15 events recorded
  10/04/2026 15:13:41 - Arquivado Definitivamente
  16/03/2026 04:23:15 - Decorrido prazo de NADIR BARBOSA DE ALMEIDA em 13/03/2026 23:59.
  10/03/2026 08:27:35 - Decorrido prazo de CONSELHO REGIONAL DE ODONTOLOGIA em 09/03/2026 23:59.
  23/02/2026 00:19:36 - Publicado Sentença em 23/02/2026.
  21/02/2026 00:29:17 - Disponibilizado no DJ Eletrônico em 20/02/2026


## Inspecting parties

In [5]:
print("Polo ativo:")
for p in df.iloc[0]["polo_ativo"]:
    print(f"  - {p['participante']}")

print()
print("Polo passivo:")
for p in df.iloc[0]["polo_passivo"]:
    print(f"  - {p['participante']}")

Polo ativo:
  - CONSELHO REGIONAL DE ODONTOLOGIA (RECLAMANTE)
  - DEBORA SERAFIM DA SILVA BARBOSA - OAB AL18880 - CPF: 124.519.994-38 (ADVOGADO)

Polo passivo:
  - NADIR BARBOSA DE ALMEIDA - CPF: 678.266.364-34 (RECLAMADO)


## Looking up multiple processes at once

In [6]:
cnjs = [
    "00584573120254058000",
    "00412666120254058100",
]
df_batch = trf5.cpopg(cnjs)
df_batch[["id_cnj", "processo", "classe", "orgao_julgador"]]

TRF5 cpopg: 100%|██████████| 2/2 [00:02<00:00,  1.12s/it]


,id_cnj,processo,classe,orgao_julgador
0,00584573120254058000,0058457-31.2025.4.05.8000,RECLAMAÇÃO PRÉ-PROCESSUAL (11875),Ambiente do Centro de Conciliação de Maceió
1,00412666120254058100,0041266-61.2025.4.05.8100,RECLAMAÇÃO PRÉ-PROCESSUAL (11875),Ambiente do Centro de Conciliação do Ceará


## Handling processes the public portal cannot return

When a CNJ does not surface in the public consultation (sigilo, archived,
or simply not found), the row carries only `id_cnj`. The other columns
come back as `None`/`NaN`, so callers can still distinguish "looked up
but missing" from "never tried".

In [7]:
import pandas as pd

df_missing = trf5.cpopg("00000000020994050000")
print("processo:", df_missing.iloc[0].get("processo"))
print("classe:", df_missing.iloc[0].get("classe"))

TRF5 cpopg: 100%|██████████| 1/1 [00:00<00:00,  8.63it/s]

processo: None
classe: None


## Splitting download from parse

`cpopg` is a thin wrapper over `cpopg_download` (raw HTML) +
`cpopg_parse` (HTML → DataFrame). Splitting them is useful when you
want to cache the raw HTMLs to disk before processing.

In [8]:
htmls = trf5.cpopg_download("0058457-31.2025.4.05.8000")
print(f"got {len(htmls)} HTML(s), {len(htmls[0])} chars")

df_again = trf5.cpopg_parse(htmls, ["00584573120254058000"])
df_again[["id_cnj", "processo", "data_distribuicao"]]

TRF5 cpopg: 100%|██████████| 1/1 [00:00<00:00,  1.48it/s]

got 1 HTML(s), 80218 chars


,id_cnj,processo,data_distribuicao
0,00584573120254058000,0058457-31.2025.4.05.8000,07/11/2025
